In [ ]:
from pathlib import Path
from collections import Counter
import json
import sqlite3

from google.colab import drive

drive.mount("/content/drive")

cache_path = Path(
    "/content/drive/MyDrive/FacialVisualProfile/cache/mtcnn_clean.sqlite"
)

if not cache_path.is_file():
    raise FileNotFoundError(cache_path)

with sqlite3.connect(
    cache_path.resolve().as_uri() + "?mode=ro",
    uri=True,
) as connection:

    print("=== Table schemas ===")
    for table in ("metadata", "detections"):
        columns = connection.execute(
            f"PRAGMA table_info({table})"
        ).fetchall()
        print(table, columns)

    print("\n=== Cache configuration ===")
    row = connection.execute(
        "SELECT value FROM metadata WHERE key = 'config'"
    ).fetchone()
    if row is None:
        raise ValueError("Cache configuration is missing")

    config = json.loads(row[0])
    print(json.dumps(config, indent=2))

    print("\n=== Counts by split and status ===")
    counts = Counter()
    examples = {}

    for name, split, encoded in connection.execute(
        "SELECT name, split, result FROM detections ORDER BY name"
    ):
        result = json.loads(encoded)
        status = result.get("status", "missing_status")
        counts[(split, status)] += 1
        examples.setdefault(status, (name, split, result))

    for (split, status), count in sorted(counts.items()):
        print(f"split={split}, status={status}: {count:,}")

    print(f"Total records: {sum(counts.values()):,}")

    print("\n=== One example per status ===")
    for status, (name, split, result) in sorted(examples.items()):
        print(f"\n{name}, split={split}, status={status}")
        print(json.dumps(result, indent=2))

In [ ]:
from pathlib import Path
import subprocess

workspace = Path("/content/day10_geometry_workspace")
expected_commit = "c4fd5b6"
repository = "https://github.com/DeweyYip/facial-visual-profile.git"

if not workspace.exists():
    subprocess.run(
        ["git", "clone", "--no-checkout", repository, str(workspace)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(workspace), "checkout", "--detach",
         expected_commit],
        check=True,
    )
else:
    # Reuse only the expected, unmodified checkout.
    status = subprocess.check_output(
        ["git", "-C", str(workspace), "status", "--porcelain"],
        text=True,
    )
    if status.strip():
        raise RuntimeError(
            "Day 10 workspace has local changes; inspect before continuing."
        )

commit = subprocess.check_output(
    ["git", "-C", str(workspace), "rev-parse", "HEAD"],
    text=True,
).strip()

if not commit.startswith(expected_commit):
    raise RuntimeError(f"Unexpected workspace commit: {commit}")

print("Workspace:", workspace)
print("Commit:", commit)

print("\n=== Required source files ===")
for name in [
    "datasets/landmark_cache.py",
    "datasets/preprocessing.py",
    "datasets/celeba.py",
    "models/resnet18_geometry.py",
]:
    path = workspace / name
    if not path.is_file():
        raise FileNotFoundError(path)
    print(name, "OK")

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
data_dir = drive_root / "data"

print("\n=== Drive data directory ===")
if not data_dir.is_dir():
    raise FileNotFoundError(data_dir)

for path in sorted(data_dir.iterdir()):
    kind = "directory" if path.is_dir() else "file"
    print(f"{path.name} ({kind})")

print("\n=== Cache file ===")
cache_path = drive_root / "cache/mtcnn_clean.sqlite"
if not cache_path.is_file():
    raise FileNotFoundError(cache_path)

print(cache_path)
print(f"Size: {cache_path.stat().st_size / 1024**2:.2f} MiB")
print("\nDay 10 source setup: PASS")

In [ ]:
from pathlib import Path
from collections import Counter
import importlib
import json
import shutil
import sys

import torch

workspace = Path("/content/day10_geometry_workspace")
drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
local_dir = Path("/content/day10_geometry_data")
local_dir.mkdir(parents=True, exist_ok=True)

# Copy inputs locally; preserve the original Drive files.
cache_path = local_dir / "mtcnn_clean.sqlite"
partition_path = local_dir / "list_eval_partition.txt"

shutil.copy2(
    drive_root / "cache/mtcnn_clean.sqlite",
    cache_path,
)
shutil.copy2(
    drive_root / "data/list_eval_partition.txt",
    partition_path,
)

# Prevent accidentally importing a different project's datasets package.
for name, module in list(sys.modules.items()):
    if name == "datasets" or name.startswith("datasets."):
        origin = getattr(module, "__file__", None)
        if origin is None or not Path(origin).resolve().is_relative_to(
            workspace.resolve()
        ):
            raise RuntimeError(
                "Another datasets package is already imported. "
                "Restart the runtime and rerun the Day 10 cells."
            )

sys.path.insert(0, str(workspace))
importlib.invalidate_caches()

import datasets.landmark_cache as cache_module
from datasets.landmark_cache import MTCNNLandmarkCache

assert Path(cache_module.__file__).resolve() == (
    workspace / "datasets/landmark_cache.py"
).resolve()
print("Reader source:", cache_module.__file__)

expected_counts = {
    "train": {"ok": 162521, "no_face": 249},
    "valid": {"ok": 19835, "no_face": 32},
}

for split, expected in expected_counts.items():
    cache = MTCNNLandmarkCache(
        cache_path,
        partition_path,
        split=split,
    )
    counts = Counter()

    for name, encoded in cache.records.items():
        record = json.loads(encoded)
        points, status = cache.get(name, image_size=(178, 218))
        counts[status] += 1

        if status == "ok":
            assert points.shape == (5, 2)
            assert points.dtype == torch.float32
            assert torch.isfinite(points).all()
            assert torch.equal(
                points,
                torch.tensor(record["landmarks"], dtype=torch.float32),
            )
        else:
            assert points is None

    assert dict(counts) == expected, dict(counts)
    print(f"{split}: {len(cache):,} records; {dict(counts)}; PASS")

    if split == "train":
        points, status = cache.get("000001.jpg", (178, 218))
        print("000001.jpg:", status)
        print(points)

        points, status = cache.get("000199.jpg", (178, 218))
        assert points is None and status == "no_face"
        print("000199.jpg:", status, points)

    del cache

print("Day 10 real-cache reader check: PASS")

In [ ]:
from io import BytesIO
import hashlib
import json
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
import torch

import datasets.preprocessing as preprocessing_module
from datasets.preprocessing import prepare, MEAN, STD, SIZE

assert Path(preprocessing_module.__file__).resolve() == (
    workspace / "datasets/preprocessing.py"
).resolve()

rng = random.Random(0)
samples = []

for split in ("train", "valid"):
    cache = MTCNNLandmarkCache(cache_path, partition_path, split=split)
    successful_names = sorted(
        name for name, encoded in cache.records.items()
        if json.loads(encoded)["status"] == "ok"
    )
    for name in rng.sample(successful_names, 3):
        samples.append((split, name, json.loads(cache.records[name])))
    del cache

archive_path = drive_root / "data/img_align_celeba.zip"
colors = ["red", "blue", "yellow", "lime", "magenta"]
fig, axes = plt.subplots(2, 3, figsize=(12, 9))

with zipfile.ZipFile(archive_path) as archive:
    members = {}
    for member in archive.namelist():
        if member.endswith("/"):
            continue
        name = Path(member).name
        if name in members:
            raise ValueError(f"Duplicate archive filename: {name}")
        members[name] = member

    for axis, (split, name, record) in zip(axes.flat, samples):
        image_bytes = archive.read(members[name])

        assert hashlib.sha256(image_bytes).hexdigest() == (
            record["image_sha256"]
        ), f"Image hash mismatch: {name}"

        with Image.open(BytesIO(image_bytes)) as image:
            assert list(image.size) == record["image_size"]
            raw = torch.tensor(
                record["landmarks"], dtype=torch.float32
            )

            image_tensor, normalized = prepare(image, raw)

            # Coordinate input must not change the image branch input.
            reference_image, _ = prepare(image, torch.zeros(10))
            assert torch.equal(image_tensor, reference_image)

            width, height = image.size

        assert image_tensor.shape == (3, 224, 224)
        assert normalized.shape == (10,)
        assert torch.isfinite(normalized).all()

        # Independently calculate the expected canvas pixel coordinates.
        scale = SIZE / max(width, height)
        new_width = round(width * scale)
        new_height = round(height * scale)
        left = (SIZE - new_width) // 2
        top = (SIZE - new_height) // 2

        expected_pixels = torch.empty_like(raw)
        expected_pixels[:, 0] = (
            (raw[:, 0] + 0.5) * new_width / width - 0.5 + left
        )
        expected_pixels[:, 1] = (
            (raw[:, 1] + 0.5) * new_height / height - 0.5 + top
        )

        canvas_points = (
            (normalized.reshape(5, 2) + 1) * (SIZE - 1) / 2
        )
        max_error = (canvas_points - expected_pixels).abs().max().item()
        assert max_error < 1e-4, (name, max_error)

        mean = torch.tensor(MEAN).view(3, 1, 1)
        std = torch.tensor(STD).view(3, 1, 1)
        display_image = (
            image_tensor * std + mean
        ).permute(1, 2, 0).clamp(0, 1).numpy()

        axis.imshow(display_image)
        axis.scatter(
            canvas_points[:, 0].numpy(),
            canvas_points[:, 1].numpy(),
            c=colors,
            s=35,
            edgecolors="black",
            linewidths=0.5,
        )
        axis.set_title(f"{split}: {name}")
        axis.axis("off")

        print(
            f"{split}/{name}: hash, size, image input, "
            f"coordinate mapping PASS; max error={max_error:.8f}px"
        )

plt.tight_layout()
plt.show()
print("Day 10 sampled image-coordinate check: PASS")

In [ ]:
import os
import subprocess
import shutil
import hashlib
import json
import zipfile
import importlib
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

workspace = Path("/content/day10_geometry_workspace")
expected_commit = "41d36c4"

status = subprocess.check_output(
    ["git", "-C", str(workspace), "status", "--porcelain"],
    text=True,
)
if status.strip():
    raise RuntimeError("Workspace has local changes; inspect first.")

# Existing imported source files must remain unchanged after checkout.
imported_hashes = {}
for module_name, module in list(sys.modules.items()):
    if module_name == "datasets" or module_name.startswith("datasets."):
        origin = getattr(module, "__file__", None)
        if origin:
            path = Path(origin).resolve()
            if not path.is_relative_to(workspace.resolve()):
                raise RuntimeError(f"Unexpected imported source: {path}")
            imported_hashes[path] = hashlib.sha256(
                path.read_bytes()
            ).hexdigest()

subprocess.run(
    ["git", "-C", str(workspace), "fetch", "origin", "main"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(workspace), "checkout", "--detach", expected_commit],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(workspace), "rev-parse", "HEAD"],
    text=True,
).strip()
assert commit.startswith(expected_commit)

for path, previous_hash in imported_hashes.items():
    if hashlib.sha256(path.read_bytes()).hexdigest() != previous_hash:
        raise RuntimeError(
            "An imported source changed; restart and rerun the notebook."
        )

importlib.invalidate_caches()
os.chdir(workspace)

from datasets.celeba import CelebAAttributes
from datasets.celeba_predicted import (
    CelebAPredictedAttributes,
    LandmarkUnavailableError,
)
from models.resnet18_geometry import ResNet18GeometryAttributes

print("Source commit:", commit)

# Full annotation files retain the official sample membership.
raw_root = Path("/content/day10_geometry_data/raw")
image_dir = raw_root / "img_align_celeba"
image_dir.mkdir(parents=True, exist_ok=True)

for name in [
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
]:
    shutil.copy2(drive_root / "data" / name, raw_root / name)

selected = {
    "train": ["101140.jpg", "110412.jpg", "010626.jpg"],
    "valid": ["171267.jpg", "179547.jpg", "178715.jpg"],
}
needed = {
    name for names in selected.values() for name in names
} | {"000199.jpg"}

with zipfile.ZipFile(
    drive_root / "data/img_align_celeba.zip"
) as archive:
    members = {}
    for member in archive.namelist():
        if member.endswith("/"):
            continue
        name = Path(member).name
        if name in needed:
            if name in members:
                raise ValueError(f"Duplicate archive filename: {name}")
            members[name] = member

    assert set(members) == needed
    for name, member in members.items():
        (image_dir / name).write_bytes(archive.read(member))

print("Locally extracted images:", len(needed))

torch.manual_seed(0)
torch.set_num_threads(2)
model = ResNet18GeometryAttributes(pretrained=False).cpu().eval()

for split, names in selected.items():
    baseline = CelebAAttributes(split=split, root=raw_root)
    dataset = CelebAPredictedAttributes(
        cache_path, split=split, root=raw_root
    )
    assert dataset.names == baseline.names

    indices = [dataset.names.index(name) for name in names]

    for index in indices:
        image, labels, points, name = dataset[index]
        base_image, base_labels, _, base_name = baseline[index]

        assert name == base_name
        assert torch.equal(image, base_image)
        assert torch.equal(labels, base_labels)
        assert points.shape == (10,)
        assert torch.isfinite(points).all()

        record = json.loads(dataset.predicted_cache.records[name])
        assert hashlib.sha256(
            (image_dir / name).read_bytes()
        ).hexdigest() == record["image_sha256"]

    print(f"{split}: sampled image/label equality and hashes PASS")

    loader = DataLoader(
        Subset(dataset, indices),
        batch_size=3,
        shuffle=False,
        num_workers=2,
    )
    images, labels, points, filenames = next(iter(loader))
    assert images.shape == (3, 3, 224, 224)
    assert labels.shape == (3, 24)
    assert points.shape == (3, 10)
    assert list(filenames) == names
    assert ((labels == 0) | (labels == 1)).all()

    with torch.inference_mode():
        logits = model(images, points)
        loss = nn.BCEWithLogitsLoss()(logits, labels)

    assert logits.shape == (3, 24)
    assert torch.isfinite(logits).all()
    assert torch.isfinite(loss)
    print(
        f"{split}: two-worker batch → model PASS; "
        f"logits={tuple(logits.shape)}, BCE={loss.item():.6f}"
    )

    if split == "train":
        failure_index = dataset.names.index("000199.jpg")
        try:
            dataset[failure_index]
        except LandmarkUnavailableError as error:
            print("Real failed detection: PASS")
            print(error)
        else:
            raise AssertionError("Expected LandmarkUnavailableError")

    del loader, baseline, dataset

print("Day 10 real Dataset/DataLoader/model integration: PASS")

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import shutil

output_dir = (
    drive_root / "runs/m3_preparation/day10"
)
source_dir = output_dir / "source"
source_dir.mkdir(parents=True, exist_ok=True)

source_hashes = {}
for name in [
    "configs/attributes.yaml",
    "datasets/celeba.py",
    "datasets/preprocessing.py",
    "datasets/landmark_cache.py",
    "datasets/celeba_predicted.py",
    "models/resnet18_geometry.py",
]:
    source_path = workspace / name
    destination = source_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination)
    source_hashes[name] = hashlib.sha256(
        source_path.read_bytes()
    ).hexdigest()

report = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_commit": commit,
    "source_sha256": source_hashes,
    "real_cache_reader": {
        "train": {"ok": 162521, "no_face": 249},
        "valid": {"ok": 19835, "no_face": 32},
        "result": "PASS",
    },
    "sampled_alignment": {
        "sample_seed": 0,
        "filenames": selected,
        "image_hash_and_size": "PASS",
        "coordinate_mapping": "PASS",
        "max_error_pixels_approx": 0.00000763,
        "visual_inspection": "No obvious misalignment in six samples",
    },
    "real_dataset_integration": {
        "sampled_baseline_image_and_label_equality": "PASS",
        "dataloader_workers": 2,
        "batch_size": 3,
        "logits_shape": [3, 24],
        "train_smoke_bce": 0.766356,
        "valid_smoke_bce": 0.942679,
        "pretrained": False,
        "model_mode": "eval",
        "failed_sample": "000199.jpg",
        "failed_sample_handling": "LandmarkUnavailableError",
        "result": "PASS",
    },
    "limitations": [
        "Only seven images were extracted locally.",
        "Image alignment and Dataset/model checks were sampled.",
        "No formal training or performance evaluation was performed.",
        "No geometric augmentation was implemented or checked.",
        "Final training-time detection-failure policy remains pending.",
    ],
}

report_path = output_dir / "integration_report.json"
report_path.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)

fig.savefig(
    output_dir / "sampled_landmark_alignment.png",
    dpi=150,
    bbox_inches="tight",
)

print("Saved:", report_path)
print("Saved:", output_dir / "sampled_landmark_alignment.png")
print("Saved source snapshot:", source_dir)
print("Day 10 verification artifacts saved.")